# Run Qwen3-32B on Kaggle (full precision on H100 80GB)

This notebook is set up for Kaggle GPU.

If you are using an **H100 80GB**, you can try loading Qwen3-32B without 4-bit quantization using `bfloat16` and `device_map="auto"`.

If memory issues appear, lower `max_new_tokens` first, then consider 8-bit or 4-bit loading.

In [1]:
# Install dependencies (Kaggle)
%pip -q install -U transformers accelerate sentencepiece fastapi uvicorn pyngrok pydantic requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 93.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not insta

In [2]:
import os
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU VRAM: {total_gb:.2f} GB')

Torch: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
GPU VRAM: 79.18 GB


In [3]:
model_name = "Qwen/Qwen3-32B"

# Full-precision-ish load for H100 (bf16)
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# H100 has 80GB, allow using most of it for model
max_memory = {0: "75GB"}  # Reserve 5GB for inference overhead

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

print(f"Model loaded successfully with dtype={compute_dtype} (no 4-bit quantization).")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully with dtype=torch.bfloat16 (no 4-bit quantization).
Model device: cuda:0
Model dtype: torch.bfloat16


In [ ]:
# # Inference
# prompt = "Give me a short introduction to large language model."
# messages = [{"role": "user", "content": prompt}]

# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True,
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# with torch.inference_mode():
#     generated_ids = model.generate(
#         **model_inputs,
#         max_new_tokens=512,
#         do_sample=True,
#         temperature=0.7,
#         top_p=0.9,
#     )

# output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
# raw_text = tokenizer.decode(output_ids, skip_special_tokens=False)

# if "</think>" in raw_text:
#     thinking_content, content = raw_text.split("</think>", 1)
#     thinking_content = thinking_content.replace("<think>", "").strip()
#     content = content.strip()
# else:
#     thinking_content = ""
#     content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

# print("thinking content:", thinking_content)
# print("content:", content)

## Model-only API in notebook (recommended)

This notebook should only run Qwen inference.

- `POST /generate` → model inference only
- `GET /health` → model status

Keep RAG and Neo4j on your own device in a separate service.
The device-side service can call this notebook API as the generator model endpoint.

In [ ]:
import threading
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field


def generate_answer(user_prompt: str, enable_thinking: bool = False, max_new_tokens: int = 512) -> dict:
    messages = [{"role": "user", "content": user_prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    raw_text = tokenizer.decode(output_ids, skip_special_tokens=False)

    if "</think>" in raw_text:
        thinking_content, content = raw_text.split("</think>", 1)
        thinking_content = thinking_content.replace("<think>", "").strip()
        content = content.strip()
    else:
        thinking_content = ""
        content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return {"thinking": thinking_content, "answer": content}


app = FastAPI(title="Qwen Model API")


class GenerateRequest(BaseModel):
    prompt: str = Field(..., min_length=1)
    max_new_tokens: int = 512
    enable_thinking: bool = False


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": model_name,
        "device": str(next(model.parameters()).device),
        "dtype": str(next(model.parameters()).dtype),
    }


@app.post("/generate")
def generate(req: GenerateRequest):
    out = generate_answer(
        user_prompt=req.prompt,
        enable_thinking=req.enable_thinking,
        max_new_tokens=req.max_new_tokens,
    )
    return {
        "prompt": req.prompt,
        "answer": out["answer"],
        "thinking": out["thinking"],
    }


def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
print("Model-only API server started on port 8000")

Model-only API server started on port 8000


INFO:     Started server process [106]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [5]:
from pyngrok import ngrok

# Optional: set token at runtime
import os
os.environ["NGROK_AUTHTOKEN"] = "2wJr14hOWIN26pFDwrn6KywARmK_57gAzAmXqGBbYckwSQ2mB"

authtoken = os.getenv("NGROK_AUTHTOKEN")
if authtoken:
    ngrok.set_auth_token(authtoken)

# Close old tunnels to avoid duplicates
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(8000, "http")
print("Public URL:", tunnel.public_url)
print("Health check:", f"{tunnel.public_url}/health")

Public URL: https://c4e0-34-13-217-185.ngrok-free.app                                               
Health check: https://c4e0-34-13-217-185.ngrok-free.app/health


In [6]:
import requests

base_url = tunnel.public_url

payload = {
    "prompt": "What is retrieval augmented generation?",
    "max_new_tokens": 256,
    "enable_thinking": False,
}

resp = requests.post(f"{base_url}/generate", json=payload, timeout=180)
print("Status:", resp.status_code)
print(resp.json())